# 3-Clustering of spots

Inés Sentís 

In [ ]:
from pyprojroot import here
here = str(here())
import sys, os

args = sys.argv[1:]
subproj = args[2]

sys.path.append(f'../../{subproj}/st_analysis/misc/')
sys.path.append(os.path.join(here,"utils"))

In [ ]:
# basic pkgs
import pandas as pd
import numpy as np
import glob
import pickle

# analysis pkgs
import scanpy as sc

# plotting pkgs
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns

# own modules
from paths import *
from module import metrics, lognorm, neighbors_umap, try_clusters, run_spatialde

%load_ext autoreload
%autoreload 2

## Define paths and variables

In [ ]:
path_to_raw = os.path.join(here, batch, "1-spaceranger/results/")
path_to_data = os.path.join(here, batch, "st_analysis/2-qc_analysis", version, "py_objects_"+version)
path_to_figures = os.path.join(here, batch, "st_analysis/3-clustering", version, "plots_"+version)

if not os.path.exists(path_to_figures):
    os.makedirs(path_to_figures)
    
sc.logging.print_versions()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.figdir = path_to_figures
sc.settings.verbosity = 0

In [ ]:
df_meta = pd.read_csv(os.path.join(here, batch, "1-spaceranger/data", "sample_id.csv"))

## Load data

In [ ]:
dict_samples = dict()

for dire in glob.glob(os.path.join(path_to_raw, '*', 'outs')):
    
    # Get GEM name
    gem = dire.split("/")[-2]
    
    if gem in df_meta['gem_id'].tolist():
        
        # Convert to sample name
        sample = df_meta[df_meta['gem_id'] == gem].reset_index().loc[0, 'donor_id']
        patient = df_meta[df_meta['gem_id'] == gem].reset_index().loc[0, 'patient']
    
        # create AnnData
        adata =  sc.read_visium(path=dire)
        adata.var_names_make_unique(join='__')
        len(adata.var_names[adata.var_names.str.contains("__")])
        adata.obs_names = [f"{patient}_{barcode}" for barcode in adata.obs_names]

        # read discarded barcode files 
        l_csv = glob.glob(os.path.join(here, batch, "st_analysis/2-qc_analysis", 
                                       version, "R_objects_"+version, "shinny", patient, 
                                       "selection-shinyapp*.csv"))
    
        bc_out = list()
        for f_csv in l_csv:
            bc_out.extend(pd.read_csv(f_csv)['barcode'].tolist())
        bc_out = list(set(bc_out))
  
        adata = adata[~adata.obs.index.isin(bc_out)]

        # add it to the dict
        dict_samples[patient] = adata

## Calculate metrics and remove genes not expressed in any of the spots

In [ ]:
dict_samples = {k:metrics(v) for (k,v) in dict_samples.items()}
dict_samples

In [ ]:
max_pct_mt_x_spot = 15 # this will be a general threshold for all of them given the plots in 02-QC_common.html

Filter and check filtered results in plot

In [ ]:
for i, (k,an) in enumerate(dict_samples.items()):
    dict_samples[k] = dict_samples[k][dict_samples[k].obs['pct_counts_mt'] < max_pct_mt_x_spot, :]

In [ ]:
metric = 'total_counts'

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16,20))

i=iter([0,1,1,0])
j=iter([1,0,1,0])

for k,an in dict_samples.items():
    sc.pl.spatial(an, img_key="hires", color=metric, ax=axes[next(i),next(j)], show=False, title=k,
                 color_map = "Spectral_r")

## Normalize and High Variable Genes

In [ ]:
dict_samples = {s:lognorm(an) for s,an in dict_samples.items()}

## Spatially variable genes

SpatialDe allows one to analyze spatial patterns of gene expression. 

In [ ]:
spatialde_results = {}

# get the spatially variable genes 
spatialde_results = {s:run_spatialde(an) for s,an in dict_samples.items()}

In [ ]:
spatialde_results.keys()

In [ ]:
for k in spatialde_results.keys():
    df_tmp = spatialde_results[k]
    df_tmp = df_tmp[df_tmp["qval"]<0.05]
    print(len(df_tmp))
    # Add SVG as HVG 
    dict_samples[k].var['highly_variable'] = dict_samples[k].var_names.isin(df_tmp["g"].tolist())

In [ ]:
# Scale the data (modifies adata.X)
for s, an in dict_samples.items():
    sc.pp.scale(an)

## Manifold embedding and clustering based on transcriptional similarity

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16,10))

i=iter([0,1,1,0])
j=iter([1,0,1,0])

for k in dict_samples.keys():
    # Compute and plot PCA
    sc.pp.pca(dict_samples[k], use_highly_variable=True, random_state=0)
    #sc.pp.pca(dict_samples[k], random_state=0)
    sc.pl.pca(dict_samples[k], ax=axes[next(i),next(j)],show=False, title=k, color ="SMARCB1")
    
    # Revert to log-normalized values
    dict_samples[k].X = dict_samples[k].raw.X.copy()
    
plt.show()

In [ ]:
fig = plt.figure(figsize=(16,10))

for i, (k,an) in enumerate(dict_samples.items()):
    sc.pl.pca_variance_ratio(an,show=False, n_pcs=50,log=True)
plt.show()

In [ ]:
dict_samples = {s:neighbors_umap(an) for s,an in dict_samples.items()}

In [ ]:
# define some resolutions 
list_reso = [0.01, 0.1, 0.5, 0.75]

# find clusters at different resolutions
dict_samples = {s:try_clusters(an, list_reso) for s,an in dict_samples.items()}

In [ ]:
# plot clusters in UMAP
for sample in dict_samples.keys():
    fig = sc.pl.umap(dict_samples[sample], color=['clusters_reso_'+str(reso) for reso in list_reso], 
           wspace=0.2, ncols=2, return_fig=True)
    
    fig.suptitle(sample, ha='center', fontsize=18)
    fig.savefig(os.path.join(path_to_figures, "UMAP_reso_"+sample+".svg"))

In [ ]:
# plot some metrics for more QC 
list_metrics_check = ["total_counts", "n_genes_by_counts","pct_counts_mt", "pct_counts_ribo"]

# plot clusters in UMAP
for sample in dict_samples.keys():
    fig = sc.pl.umap(dict_samples[sample], color=list_metrics_check, color_map = "Spectral_r",
           wspace=0.2, ncols=2, return_fig=True)
    
    fig.suptitle(sample, ha='center', fontsize=18)
    fig.savefig(os.path.join(path_to_figures, "UMAP_checks_"+sample+".svg"))

## Try more resolutions and plot them into tissue section

In [ ]:
# define some more resolutions 
list_reso_extra = [0.25, 0.35]

# find clusters at different resolutions
dict_samples = {s:try_clusters(an, list_reso_extra) for s,an in dict_samples.items()}

# plot clusters in UMAP
for sample in dict_samples.keys():
    fig = sc.pl.umap(dict_samples[sample], color=['clusters_reso_'+str(reso) for reso in list_reso_extra], 
           wspace=0.2, ncols=2, return_fig=True)
    
    fig.suptitle(sample, ha='center', fontsize=18)
    fig.savefig(os.path.join(path_to_figures, "UMAP_reso_extra_"+sample+".svg"))

In [ ]:
list_reso.extend(list_reso_extra)
list_reso = list(set(list_reso))
list_reso.sort(key= lambda x: x)

# plot clusters at different resolutions  
for k,an in dict_samples.items():
    
    # create figure of mutiple subplots with scanpy.pl.spatial
    sc.set_figure_params(fontsize=20)
    sc.pl.spatial(an, img_key="hires", 
              color=['clusters_reso_'+str(reso) for reso in list_reso], ncols=3,
              use_raw=False, save='_'+k+'_reso.svg')

In [ ]:
plt.rcParams["figure.figsize"] = (10,7)
sc.set_figure_params(fontsize=12)
for sample in dict_samples.keys():
    sc.pl.violin(
        dict_samples[sample],
        keys = ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
        groupby='clusters_reso_0.35',
        jitter=0.4,
        multi_panel=True,
    )

## Save objects

In [ ]:
path_out = os.path.join(here, batch, "st_analysis/3-clustering", version, "py_objects_"+version)

if not os.path.exists(path_out):
    os.mkdir(path_out)
    
for k,an in dict_samples.items():
    an.write(os.path.join(path_out, k+'.h5ad'))

In [ ]:
# store results from SpatialDe in a pickle
with open(os.path.join(path_out, 'results_spatialde.pkl'), 'wb') as handle:
    pickle.dump(spatialde_results, handle, protocol=pickle.HIGHEST_PROTOCOL)